In [1]:
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import pandas as pd

nltk.download('vader_lexicon')

df = pd.read_csv("spotify_millsongdata.csv") #main dataset

additional_df = pd.read_csv("additional_features.csv") #includes features like danceability,valence, etc

vds = SentimentIntensityAnalyzer()

#gets the vader sentiment analysis score for each song
records = []
for _, row in df.iterrows():
    scores = vds.polarity_scores(str(row["text"]))
    records.append({
        "Song":     row["song"],
        "Artist":   row["artist"],
        "Compound": scores["compound"],
        "Negative":      scores["neg"],
        "Neutral":      scores["neu"],
        "Positive":      scores["pos"],
    })

#make new dataset with polarity scores appended
sentiment_df = pd.DataFrame(records)

#dataset with songs, polarity scores, and additional features
merged_df = pd.merge(sentiment_df, additional_df, how="inner", left_on="Song", right_on="track_name")

# Drop duplicate song name column if needed
merged_df.drop(columns=["Unnamed: 0", "track_id", "duration_ms", "time_signature", "track_name", "artists", "album_name"], inplace=True)
merged_df = merged_df.drop_duplicates(subset=['Song'], keep='first')

# Save final merged dataset to CSV
merged_df.to_csv("merged_spotify_dataset.csv", index=False)

print("Merged dataset saved as 'merged_spotify_dataset.csv'")

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/shayna/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


Merged dataset saved as 'merged_spotify_dataset.csv'


In [4]:
import torch
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

merged_df = pd.read_csv("merged_spotify_dataset.csv") 

#initialize embedder and regressor
device    = "cuda" if torch.cuda.is_available() else "cpu"
#loading pre trained sentence embedding model here. this converts english text to vectors.
embedder  = SentenceTransformer("BAAI/bge-large-en-v1.5", device=device)
regressor = Ridge()

#build text descriptions for each track. 
#the embedding model only takes in text, so we translate the charactersitics of evrey to a short english sentence.
#note: may need to tweak this description a bit
def row_to_description(row):
    return (f"A {row['track_genre']} song called '{row['Song']}' by {row['Artist']} "
            f"that is {'explicit' if row['explicit'] else 'not explicit'}, "
            f"with popularity {row['popularity']:.0f}, "
            f"{'high' if row['energy']>0.6 else 'low'} energy, "
            f"and {'high' if row['acousticness']>0.6 else 'low'} acousticness.")

descriptions = merged_df.apply(row_to_description, axis=1).tolist()

#takes in those short english sentences for every song and computes embedding for the key words described in the description like explicit, popularity, energy, acousticness.
X_embeddings = embedder.encode(
    descriptions,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

#train a Ridge regressor on true Spotify features
#selecting spotify features we'll try to predict
#map the embedding above for explicit, popularity, energy, acousticness to the actual features in this dataset.
#this is just for us to see but its not fed into kmeans in any way
feature_cols = [
    'Compound','Negative','Neutral','Positive',
    'danceability','energy','key','loudness',
    'mode','speechiness','acousticness',
    'instrumentalness','liveness','valence','tempo'
]
y = merged_df[feature_cols].values

X_train, X_test, y_train, y_test = train_test_split(
    X_embeddings, y, test_size=0.2, random_state=42
)
regressor.fit(X_train, y_train)

#kmeans on embeddings
optimal_k = 5
kmeans = KMeans(n_clusters=optimal_k, random_state=42).fit(X_embeddings)

Batches: 100%|██████████| 87/87 [01:41<00:00,  1.16s/it]


In [5]:
#dynamic query processing

import numpy as np
from sentence_transformers import SentenceTransformer

#embed the users input
def process_user_query(query_text, k=10):
    query_emb = embedder.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    
    pred_feats = regressor.predict(query_emb)[0]
    print("Predicted feature vector:")
    for name, val in zip(feature_cols, pred_feats):
        print(f"  {name}: {val:.3f}")
    
    cluster_label = kmeans.predict(query_emb)[0]
    print(f"\nAssigned to cluster: {cluster_label}")
    
    cluster_idxs = np.where(kmeans.labels_ == cluster_label)[0]
    dists = np.linalg.norm(X_embeddings[cluster_idxs] - query_emb, axis=1)
    nearest = cluster_idxs[np.argsort(dists)[:k]]
    neighbors = merged_df.iloc[nearest][['Song','Artist']].reset_index(drop=True)
    
    print(f"\nTop {k} nearest songs in the same cluster:")
    print(neighbors)

while True:
    q = input("Describe the kind of music you're in the mood for: (enter 'quit' to exit)")
    if q.lower() == 'quit':
        break
    process_user_query(q, k=10)


Predicted feature vector:
  Compound: 0.724
  Negative: 0.052
  Neutral: 0.712
  Positive: 0.236
  danceability: 0.701
  energy: 0.847
  key: 5.191
  loudness: -7.152
  mode: 0.440
  speechiness: 0.147
  acousticness: 0.274
  instrumentalness: -0.043
  liveness: 0.273
  valence: 0.645
  tempo: 126.625

Assigned to cluster: 1

Top 10 nearest songs in the same cluster:
                Song        Artist
0     It Gets Better          Fun.
1       We Are Young          Fun.
2               Move    Little Mix
3    Towards The Sun       Rihanna
4         Two Hearts  Phil Collins
5   The Logical Song    Supertramp
6         Good Times  Mariah Carey
7  Put Your Hands Up          Inna
8     I'm So Excited          P!nk
9     Endless Summer          Zwan
